In [15]:
# =============================================================================
# MiCRM Dataset Generator  —  v7b  (Suppression-Variance Fix)
# =============================================================================
#
# WHAT CHANGED FROM v7 AND WHY
# ─────────────────────────────
#
# v7 PROBLEM: near-all suppression values cluster at 0.99+
#
#   Root cause: three compounding factors pushed suppression to saturation.
#
#   1. DOMINANT NUTRIENTS TOO STRONG (4–10× background)
#      Generalists (40–65% uptake breadth) get an enormous energy surplus
#      from 2–4 dominants each at 4–10×.  By T=20 they have fully driven
#      the pathogen to near-zero in every large community, regardless of
#      composition — the variance the GNN needs to learn from disappears.
#
#   2. TOO MANY ABSENT NUTRIENTS (3–5)
#      With 3–5 nutrients absent, pathogen has very few accessible niches
#      left to sustain itself against the community pressure.
#
#   3. PATHOGEN UPTAKE BREADTH TOO NARROW (30–55%)
#      The pathogen competes across fewer axes than generalists, making it
#      easy to overwhelm once energy is abundant.
#
# v7b FIXES (minimal — only broth design and pathogen parameters change):
#
#   FIX 1 — Weaker dominants: 1.5–3.0× background (was 4–10×)
#      This reduces the energy surplus available to the community, keeping
#      suppression in a competitive regime rather than saturation.
#      Multiple dominants (2–3) are kept for broth discriminability.
#
#   FIX 2 — Fewer absents: 1–2 (was 3–5)
#      Pathogen retains some nutrient refuge, so suppression is not
#      automatic even for large communities.
#
#   FIX 3 — Stronger pathogen: uptake breadth 0.45–0.70 (was 0.30–0.55)
#      Pathogen now competes across more nutrient axes, making it harder
#      to fully suppress and increasing variance in the outcome.
#
#   FIX 4 — Lower pathogen maintenance: Uniform(0.05, 0.15) (was 0.10–0.50)
#      Pathogen can survive at lower energy intake, creating a genuine
#      suppression gradient rather than binary live/dead outcomes.
#
#   FIX 5 — T_MEASURE reduced to 14 (was 20)
#      Measuring earlier catches communities mid-competition, before the
#      dominant generalists have had time to fully saturate suppression.
#      The pathogen is still alive and variance is maximal at T~14.
#
#   Everything else (community sizes, sampling, cross-feeding, ODE, all
#   non-pathogen species traits, output format) is IDENTICAL to v7.
#
# Target distribution: mean ≈ 0.55–0.70, std ≈ 0.18–0.22, <25% at >0.95
# =============================================================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from scipy.integrate import solve_ivp
from tqdm import tqdm
import os

# ── Reproducibility ──────────────────────────────────────────────────────────
rng = np.random.default_rng(seed=42)

# ── Global constants (same as v6) ─────────────────────────────────────────────
N_SPECIES    = 101
N_RESOURCES  = 20
N_GENES      = 8
N_BROTHS     = 10

PATHOGEN_IDX = 0
N_COMMUNITY  = N_SPECIES - 1
N_SPECIALIST = N_COMMUNITY // 2
N_GENERALIST = N_COMMUNITY - N_SPECIALIST

T_MEASURE = 14.0   # v7b: reduced from 20 — catches mid-competition variance

# ── v7 KEY CHANGE: Large communities only ─────────────────────────────────────
COMMUNITY_SIZES    = [12, 16, 24, 32, 48]   # no small communities
N_SAMPLES_PER_SIZE = 50                     # 10 × 5 × 50 = 2500 communities

OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\GNN_Nutrient_v1_Data"    # change to your output path

# ── Sparsity (same as v6) ─────────────────────────────────────────────────────
UPTAKE_SPARSITY = {
    "pathogen"   : (0.45, 0.70),   # v7b: wider — pathogen competes across more axes
    "specialist" : (0.10, 0.25),
    "generalist" : (0.40, 0.65),
}
CROSSFEED_OUT_MIN = 2
CROSSFEED_OUT_MAX = 6

RESOURCE_NAMES = [f"c{k+1}"            for k in range(N_RESOURCES)]
SPECIES_NAMES  = [f"Species_{i+1:04d}" for i in range(N_SPECIES)]
BROTH_NAMES    = [f"Broth_{b+1:02d}"   for b in range(N_BROTHS)]

In [9]:
# =============================================================================
# 1.  SPECIES TRAITS  (identical to v6)
# =============================================================================
def build_species_traits():
    genes = rng.integers(0, 2, size=(N_SPECIES, N_GENES))
    W     = rng.normal(0.0, 0.5, size=(N_GENES, N_RESOURCES))

    c_matrix     = np.zeros((N_SPECIES, N_RESOURCES))
    uptake_mask  = np.zeros((N_SPECIES, N_RESOURCES), dtype=bool)
    species_type = []
    preferred_nutrient = []

    for i in range(N_SPECIES):
        if i == PATHOGEN_IDX:
            stype = "pathogen";  pref = -1
        elif 1 <= i <= N_SPECIALIST:
            stype = "specialist"; pref = (i - 1) % N_RESOURCES
        else:
            stype = "generalist"; pref = -1

        species_type.append(stype)
        preferred_nutrient.append(pref)

        lo, hi   = UPTAKE_SPARSITY[stype]
        n_active = max(1, int(round(rng.uniform(lo, hi) * N_RESOURCES)))

        active_indices = rng.choice(N_RESOURCES, size=n_active, replace=False).tolist()
        if pref >= 0 and pref not in active_indices:
            active_indices[0] = pref
        uptake_mask[i, active_indices] = True

        if stype == "specialist":
            bias = rng.uniform(-2.0, -1.0, size=N_RESOURCES)
            bias[pref] += rng.uniform(2.5, 4.0)
        else:
            bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)

        raw          = genes[i].astype(float) @ W + bias
        c_matrix[i]  = (1.0 / (1.0 + np.exp(-raw))) * uptake_mask[i]

    return genes, c_matrix, uptake_mask, species_type, preferred_nutrient

In [10]:
# =============================================================================
# 2.  BROTH ENVIRONMENTS  — v7 redesign
# =============================================================================
def build_broths():
    """
    v7b broth design: multiple discriminative dominants, but weaker intensity.

    For each broth:
      • n_dominant = 2–3 dominant nutrients, each at 1.5–3.0× background.
        Multiple dominants keep broth fingerprints distinct for the GNN;
        the reduced multiplier prevents energy saturation in large communities.
      • n_absent   = 1–2 nutrients explicitly set to 0.
        Fewer absents give the pathogen some nutrient refuge so suppression
        does not become automatic in every large community.
      • background = Uniform(0.5, 1.5) for remaining nutrients.
      • delta, w, l unchanged from v7.
    """
    broths = []

    for b in range(N_BROTHS):
        # ── Sample dominant nutrients ─────────────────────────────────────────
        n_dominant   = int(rng.integers(2, 4))      # 2 or 3 dominants
        dominant_set = set(
            rng.choice(N_RESOURCES, size=n_dominant, replace=False).tolist()
        )

        # ── Sample absent nutrients (from non-dominant only) ──────────────────
        non_dom  = [k for k in range(N_RESOURCES) if k not in dominant_set]
        n_absent = int(rng.integers(1, 3))           # 1 or 2 absents (v7b: reduced)
        n_absent = min(n_absent, len(non_dom))
        absent_set = set(
            rng.choice(non_dom, size=n_absent, replace=False).tolist()
        )

        # ── Build R0 vector ───────────────────────────────────────────────────
        R0 = np.zeros(N_RESOURCES)
        for k in range(N_RESOURCES):
            if k in dominant_set:
                # v7b: weaker multiplier 1.5–3.0× (was 4–10× in v7)
                R0[k] = rng.uniform(0.5, 1.5) * rng.uniform(1.5, 3.0)
            elif k in absent_set:
                R0[k] = 0.0
            else:
                R0[k] = rng.uniform(0.5, 1.5)

        delta = float(rng.uniform(0.05, 0.20))
        w     = rng.uniform(0.6, 1.2, size=N_RESOURCES)
        l     = rng.uniform(0.1, 0.3,  size=N_RESOURCES)

        # ── Extra feature columns (used by GNN v2) ────────────────────────────
        total_R0      = R0.sum() + 1e-12
        frac_R0       = R0 / total_R0
        dominant_flag = np.array([1 if k in dominant_set else 0
                                   for k in range(N_RESOURCES)], dtype=int)
        absent_flag   = np.array([1 if k in absent_set   else 0
                                   for k in range(N_RESOURCES)], dtype=int)

        broths.append({
            "name"          : BROTH_NAMES[b],
            "delta"         : delta,
            "R0"            : R0,
            "frac_R0"       : frac_R0,
            "dominant_flag" : dominant_flag,
            "absent_flag"   : absent_flag,
            "w"             : w,
            "l"             : l,
            "dominant_set"  : sorted(dominant_set),
            "absent_set"    : sorted(absent_set),
        })

    return broths

In [11]:
# =============================================================================
# 3.  CROSS-FEEDING MATRICES  (identical to v6)
# =============================================================================
def build_species_crossfeed_matrices(species_type):
    D_list = []
    for i in range(N_SPECIES):
        D_i = np.zeros((N_RESOURCES, N_RESOURCES))
        for beta in range(N_RESOURCES):
            n_out        = int(rng.integers(CROSSFEED_OUT_MIN, CROSSFEED_OUT_MAX + 1))
            active_alpha = rng.choice(N_RESOURCES, size=n_out, replace=False)

            if species_type[i] == "specialist":
                pref = (i - 1) % N_RESOURCES
                alpha_vec = np.array([
                    rng.uniform(2.0, 5.0) if a == pref and beta != pref
                    else rng.uniform(0.5, 2.0)
                    for a in active_alpha
                ])
            elif species_type[i] == "pathogen":
                alpha_vec = rng.uniform(0.5, 3.0, size=n_out)
            else:
                alpha_vec = rng.uniform(0.8, 1.5, size=n_out)

            probs = rng.dirichlet(alpha_vec)
            D_i[active_alpha, beta] = probs
        D_list.append(D_i)
    return D_list

In [12]:
# =============================================================================
# 4.  ODE  (identical to v6)
# =============================================================================
def micrm_ode(t, y, c, w, l, m, g_rate, delta, R0, D_list_sub):
    n_sp  = c.shape[0]
    n_res = c.shape[1]
    N = np.maximum(y[:n_sp], 0.0)
    R = np.maximum(y[n_sp:], 0.0)

    energy_intake = c @ (w * (1.0 - l) * R)
    net_growth    = g_rate * (energy_intake - m)
    dN = N * net_growth - delta * N

    dilution_net = delta * (R0 - R)
    uptake       = (c.T @ N) * R

    crossfeed = np.zeros(n_res)
    for i in range(n_sp):
        Ni = N[i]
        if Ni < 1e-14:
            continue
        for beta in range(n_res):
            if c[i, beta] < 1e-14:
                continue
            flux_i_beta = l[beta] * c[i, beta] * Ni * R[beta]
            if flux_i_beta < 1e-16:
                continue
            col = D_list_sub[i][:, beta]
            nz  = np.nonzero(col)[0]
            for alpha in nz:
                crossfeed[alpha] += col[alpha] * w[beta] / (w[alpha] + 1e-12) * flux_i_beta

    dR = dilution_net - uptake + crossfeed
    return np.concatenate([dN, dR])


def run_micrm(c_full, w, l, m_full, g_full, delta, R0, D_list,
              species_subset, t_end=T_MEASURE):
    c_sub      = c_full[species_subset, :]
    m_sub      = m_full[species_subset]
    g_sub      = g_full[species_subset]
    D_list_sub = [D_list[i] for i in species_subset]

    y0  = np.concatenate([np.full(len(species_subset), 0.01), R0.copy()])
    sol = solve_ivp(
        fun    = micrm_ode,
        t_span = (0.0, t_end),
        y0     = y0,
        args   = (c_sub, w, l, m_sub, g_sub, delta, R0, D_list_sub),
        method = "RK45",
        t_eval = np.linspace(0, t_end, max(50, int(t_end * 5))),
        rtol   = 1e-5,
        atol   = 1e-7,
    )
    y_final = sol.y[:, -1]
    N_out   = np.zeros(N_SPECIES)
    for local_idx, global_idx in enumerate(species_subset):
        N_out[global_idx] = max(0.0, y_final[local_idx])
    R_final = np.maximum(y_final[len(species_subset):], 0.0)
    return N_out, R_final, sol

In [13]:
# =============================================================================
# 5.  SUPPRESSION  (identical to v6)
# =============================================================================
def calc_suppression(N_path_measured, N_path_solo):
    eps = 1e-12
    if N_path_solo < eps:
        return 0.0
    return float(np.clip(1.0 - N_path_measured / (N_path_solo + eps), 0.0, 1.0))

def _lum_transform(supp_score, N_path_solo):
    ratio = max(1.0 - supp_score, 1e-10)
    return round(float(np.clip(np.log10(ratio), -10.0, 0.0)), 6)

In [ ]:
# =============================================================================
# 6.  MAIN DATASET GENERATION
# =============================================================================
def generate_dataset():
    print("=" * 65)
    print("  MiCRM v7b — Suppression-Variance Fix")
    print(f"  T_MEASURE = {T_MEASURE}  (reduced from 20 to catch mid-competition)")
    print(f"  Community sizes: {COMMUNITY_SIZES}  (large-only, N≥12)")
    print(f"  Samples per size per broth: {N_SAMPLES_PER_SIZE}")
    expected = N_BROTHS * len(COMMUNITY_SIZES) * N_SAMPLES_PER_SIZE
    print(f"  Expected communities: {expected}")
    print(f"  Pathogen: wider uptake (0.45-0.70), lower maintenance (0.05-0.12)")
    print(f"  Broths  : 2-3 dominants at 1.5-3x, 1-2 absents")
    print("=" * 65)

    # ── Build shared ecology ──────────────────────────────────────────────────
    print("\n[1] Building species traits ...")
    genes, c_matrix, uptake_mask, species_type, preferred_nutrient = build_species_traits()

    print("[2] Building broth environments (v7 multi-dominant design) ...")
    broths = build_broths()
    for b in broths:
        print(f"   {b['name']:10s}  dominants={[f'c{k+1}' for k in b['dominant_set']]}"
              f"  absent={[f'c{k+1}' for k in b['absent_set']]}"
              f"  R0_sum={b['R0'].sum():.1f}")

    print("[3] Building cross-feeding matrices ...")
    D_list = build_species_crossfeed_matrices(species_type)

    m = rng.uniform(0.1, 0.5, size=N_SPECIES)
    g = rng.uniform(0.5, 1.5, size=N_SPECIES)
    # v7b: pathogen has lower maintenance — harder to fully suppress
    m[PATHOGEN_IDX] = float(rng.uniform(0.05, 0.12))

    # ── Pathogen solo ─────────────────────────────────────────────────────────
    print(f"\n[4] Pathogen solo abundances ...")
    pathogen_solo = {}
    for broth in tqdm(broths, desc="  Pathogen solo", unit="broth"):
        N_s, _, _ = run_micrm(c_matrix, broth["w"], broth["l"], m, g,
                               broth["delta"], broth["R0"], D_list,
                               species_subset=[PATHOGEN_IDX])
        pathogen_solo[broth["name"]] = float(N_s[PATHOGEN_IDX])

    # ── Pairwise ss_lum ───────────────────────────────────────────────────────
    print(f"\n[5] Single-strain suppression (ss_lum) ...")
    ss_lum = {}
    strain_indices = list(range(1, N_SPECIES))
    for broth in tqdm(broths, desc="  Pairwise ODE", unit="broth"):
        bname    = broth["name"]
        N_solo_p = pathogen_solo[bname]
        ss_lum[bname] = {}
        for sp_idx in strain_indices:
            N_pair, _, _ = run_micrm(c_matrix, broth["w"], broth["l"], m, g,
                                      broth["delta"], broth["R0"], D_list,
                                      species_subset=[sp_idx, PATHOGEN_IDX])
            ss_lum[bname][sp_idx] = calc_suppression(N_pair[PATHOGEN_IDX], N_solo_p)

    # ── Community suppression ─────────────────────────────────────────────────
    print(f"\n[6] Community suppression (large communities only) ...")
    community_rows   = []
    community_id_ctr = 0

    for broth in tqdm(broths, desc="  Community ODE", unit="broth"):
        bname    = broth["name"]
        N_solo_p = pathogen_solo[bname]

        for comm_size in COMMUNITY_SIZES:
            for _ in range(N_SAMPLES_PER_SIZE):
                members    = sorted(
                    rng.choice(strain_indices, size=comm_size, replace=False).tolist()
                )
                N_comm, _, _ = run_micrm(
                    c_matrix, broth["w"], broth["l"], m, g,
                    broth["delta"], broth["R0"], D_list,
                    species_subset=members + [PATHOGEN_IDX]
                )
                comm_supp = calc_suppression(N_comm[PATHOGEN_IDX], N_solo_p)
                comm_id   = f"Comm_{community_id_ctr:05d}"
                community_id_ctr += 1

                for sp_idx in members:
                    row = {
                        # ── Community metadata ──────────────────────────────
                        "community_id"           : comm_id,
                        "broth_id"               : bname,
                        "community_size"         : comm_size,
                        "community_suppression"  : round(comm_supp, 6),

                        # ── Broth metadata ──────────────────────────────────
                        "dominant_nutrients"     : str([f"c{k+1}" for k in broth["dominant_set"]]),
                        "absent_nutrients"       : str([f"c{k+1}" for k in broth["absent_set"]]),
                        "delta_per_h"            : round(broth["delta"], 4),

                        # ── Strain identity ─────────────────────────────────
                        "strain_id"              : SPECIES_NAMES[sp_idx],
                        "species_type"           : species_type[sp_idx],
                        "maintenance_cost"       : round(float(m[sp_idx]), 4),
                        "growth_rate"            : round(float(g[sp_idx]), 4),
                        "ss_luminescence"        : _lum_transform(
                            ss_lum[bname][sp_idx], pathogen_solo[bname]
                        ),
                        "ss_suppression_raw"     : round(ss_lum[bname][sp_idx], 6),
                    }

                    # ── Genes ────────────────────────────────────────────────
                    for g_idx in range(N_GENES):
                        row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])

                    # ── Uptake rates ─────────────────────────────────────────
                    for k in range(N_RESOURCES):
                        row[f"uptake_c{k+1}"] = round(float(c_matrix[sp_idx, k]), 6)

                    # ── R0: raw initial concentrations ───────────────────────
                    for k in range(N_RESOURCES):
                        row[f"R0_c{k+1}"] = round(float(broth["R0"][k]), 4)

                    # ── v7 EXTRA: fractional & binary nutrient features ───────
                    # These give the GNN richer node features (see GNN v2 code)
                    for k in range(N_RESOURCES):
                        row[f"frac_c{k+1}"]      = round(float(broth["frac_R0"][k]), 6)
                        row[f"dominant_c{k+1}"]   = int(broth["dominant_flag"][k])
                        row[f"absent_c{k+1}"]     = int(broth["absent_flag"][k])

                    community_rows.append(row)

    df = pd.DataFrame(community_rows)
    print(f"\n  Total rows (strain × community): {len(df)}")
    print(f"  Unique communities             : {df['community_id'].nunique()}")
    print(f"\n  community_suppression summary:")
    print(df["community_suppression"].describe().round(4))

    return df, genes, c_matrix, uptake_mask, m, g, broths, D_list, species_type, ss_lum, pathogen_solo


# =============================================================================
# 7.  COMMUNITY INDEX FILE  (one row per community)
# =============================================================================
def build_community_index(df):
    idx = df.groupby("community_id").agg(
        broth_id              = ("broth_id",             "first"),
        community_size        = ("community_size",        "first"),
        community_suppression = ("community_suppression", "first"),
        dominant_nutrients    = ("dominant_nutrients",    "first"),
        absent_nutrients      = ("absent_nutrients",      "first"),
        n_members             = ("strain_id",             "count"),
    ).reset_index()
    return idx


# =============================================================================
# 8.  MONOCULTURE DATASET  (one row per strain, for reference)
# =============================================================================
def build_monoculture_dataset(df, genes, c_matrix, m, g, broths,
                               D_list, species_type, ss_lum, pathogen_solo):
    print("\n[7] Building monoculture dataset ...")
    ref_broth = broths[0]
    mono_rows = []
    for sp_idx in range(1, N_SPECIES):
        lum_vals = [_lum_transform(ss_lum[b["name"]][sp_idx], pathogen_solo[b["name"]])
                    for b in broths]
        row = {
            "strain_id"        : SPECIES_NAMES[sp_idx],
            "species_type"     : species_type[sp_idx],
            "maintenance_cost" : round(float(m[sp_idx]), 4),
            "growth_rate"      : round(float(g[sp_idx]), 4),
            "ss_luminescence"  : round(float(np.mean(lum_vals)), 6),
        }
        for g_idx in range(N_GENES):
            row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])
        for k in range(N_RESOURCES):
            row[f"uptake_c{k+1}"] = round(float(c_matrix[sp_idx, k]), 6)
            row[f"R0_c{k+1}"]     = round(float(ref_broth["R0"][k]), 4)
        mono_rows.append(row)

    df_mono = pd.DataFrame(mono_rows)
    print(f"  Monoculture dataset: {len(df_mono)} rows × {len(df_mono.columns)} cols")
    return df_mono

In [16]:
# =============================================================================
# 9.  RUN
# =============================================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df, genes, c_matrix, uptake_mask, m, g, broths, D_list, \
        species_type, ss_lum, pathogen_solo = generate_dataset()

    df_index = build_community_index(df)
    df_mono  = build_monoculture_dataset(df, genes, c_matrix, m, g, broths,
                                          D_list, species_type, ss_lum, pathogen_solo)

    # ── Save ──────────────────────────────────────────────────────────────────
    comm_path  = os.path.join(OUTPUT_DIR, "community_dataset_v7b.csv")
    index_path = os.path.join(OUTPUT_DIR, "community_index_v7b.csv")
    mono_path  = os.path.join(OUTPUT_DIR, "monoculture_dataset_v7b.csv")

    df.to_csv(comm_path,  index=False)
    df_index.to_csv(index_path, index=False)
    df_mono.to_csv(mono_path,   index=False)

    print(f"\n  Saved:")
    print(f"    {comm_path}")
    print(f"    {index_path}")
    print(f"    {mono_path}")

    # ── Quick diagnostic ──────────────────────────────────────────────────────
    print(f"\n  Between-broth suppression means (should vary meaningfully):")
    print(df_index.groupby("broth_id")["community_suppression"]
          .mean().round(3).to_string())
    bw_std = df_index.groupby("broth_id")["community_suppression"].mean().std()
    wi_std = df_index.groupby("broth_id")["community_suppression"].std().mean()
    frac_hi = (df_index["community_suppression"] > 0.95).mean()
    print(f"\n  Between-broth std : {bw_std:.4f}  (target >0.08)")
    print(f"  Within-broth  std : {wi_std:.4f}  (residual composition effect)")
    print(f"  Signal-to-noise   : {bw_std / (wi_std + 1e-8):.3f}  (target >0.4)")
    print(f"  Fraction > 0.95   : {frac_hi:.3f}  (target <0.25)")

    print("\n  Done.")

  MiCRM v7b — Suppression-Variance Fix
  T_MEASURE = 14.0  (reduced from 20 to catch mid-competition)
  Community sizes: [12, 16, 24, 32, 48]  (large-only, N≥12)
  Samples per size per broth: 50
  Expected communities: 2500
  Pathogen: wider uptake (0.45-0.70), lower maintenance (0.05-0.12)
  Broths  : 2-3 dominants at 1.5-3x, 1-2 absents

[1] Building species traits ...
[2] Building broth environments (v7 multi-dominant design) ...
   Broth_01    dominants=['c2', 'c17']  absent=['c3']  R0_sum=20.3
   Broth_02    dominants=['c7', 'c9', 'c16']  absent=['c15']  R0_sum=26.0
   Broth_03    dominants=['c7', 'c16', 'c20']  absent=['c19']  R0_sum=22.1
   Broth_04    dominants=['c10', 'c20']  absent=['c14', 'c17']  R0_sum=17.6
   Broth_05    dominants=['c4', 'c5', 'c19']  absent=['c6', 'c14']  R0_sum=19.1
   Broth_06    dominants=['c5', 'c12']  absent=['c2', 'c16']  R0_sum=20.3
   Broth_07    dominants=['c1', 'c17', 'c19']  absent=['c3', 'c13']  R0_sum=19.4
   Broth_08    dominants=['c1', 'c2'

  Pathogen solo: 100%|██████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 20.97broth/s]



[5] Single-strain suppression (ss_lum) ...


  Pairwise ODE: 100%|███████████████████████████████████████████████████████████████| 10/10 [01:22<00:00,  8.29s/broth]



[6] Community suppression (large communities only) ...


  Community ODE: 100%|█████████████████████████████████████████████████████████████| 10/10 [22:20<00:00, 134.05s/broth]



  Total rows (strain × community): 66000
  Unique communities             : 2500

  community_suppression summary:
count    66000.0000
mean         0.3876
std          0.1575
min          0.0000
25%          0.2717
50%          0.3681
75%          0.4932
max          0.8087
Name: community_suppression, dtype: float64

[7] Building monoculture dataset ...
  Monoculture dataset: 100 rows × 53 cols

  Saved:
    C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\GNN_Nutrient_v1_Data\community_dataset_v7b.csv
    C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\GNN_Nutrient_v1_Data\community_index_v7b.csv
    C:\Users\Pratham Shah\Downloads\Document dataset\Final_MyData\GNN_Nutrient_v1_Data\monoculture_dataset_v7b.csv

  Between-broth suppression means (should vary meaningfully):
broth_id
Broth_01    0.328
Broth_02    0.305
Broth_03    0.258
Broth_04    0.228
Broth_05    0.464
Broth_06    0.286
Broth_07    0.489
Broth_08    0.424
Broth_09    0.274
Broth_10    0.395